# Extension Principle and Fuzzy Arithmetic

The **extension principle** lets us apply an ordinary function to fuzzy quantities.

Suppose $A$ and $B$ are fuzzy sets and we want to compute

$z=f(x,y)$.

The extension principle defines the membership of an output value $z$ by considering **every pair of inputs that can produce that value**.

Using the minimum t-norm, the discrete form used in this notebook is

$\mu_C(z)=\max_{f(x,y)=z}\min(\mu_A(x),\mu_B(y))$.

In words:

1. find input pairs $(x,y)$ that produce $z$;
2. combine their memberships using `min`;
3. keep the largest resulting membership.

We will first implement this idea directly, then solve fuzzy arithmetic using **$\alpha$-cuts**.


## Learning objectives

By the end of this notebook, you should be able to:

- explain the extension principle in words;
- compute the image of two discrete fuzzy sets under a binary function;
- distinguish between applying `max` to **values** and taking the union of fuzzy sets;
- interpret fuzzy addition, multiplication, subtraction, and division;
- explain how $\alpha$-cuts turn fuzzy arithmetic into interval arithmetic;
- compare the direct extension-principle approach with the $\alpha$-cut approach.


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt


## 1. Two discrete fuzzy numbers

We will use the universe $X=\{1,2,\ldots,29\}$.

The first fuzzy number is **about 3**:

- $\mu_A(2)=0.5$
- $\mu_A(3)=1$
- $\mu_A(4)=0.5$

The second fuzzy number is **about 7**:

- $\mu_B(6)=0.5$
- $\mu_B(7)=1$
- $\mu_B(8)=0.5$

Every other membership value is zero.

### Prediction

If $A$ is “about 3” and $B$ is “about 7,” where should **$A+B$** have its highest membership?


In [ ]:
# Universe
X = np.arange(1, 30)

# Fuzzy number A: "about 3"
A = np.zeros(X.size)
A[1] = 0.5   # x = 2
A[2] = 1.0   # x = 3
A[3] = 0.5   # x = 4

# Fuzzy number B: "about 7"
B = np.zeros(X.size)
B[5] = 0.5   # x = 6
B[6] = 1.0   # x = 7
B[7] = 0.5   # x = 8

plt.figure(figsize=(10, 4))
plt.stem(X, A, linefmt="C0-", markerfmt="C0o", basefmt=" ", label='A: "about 3"')
plt.stem(X, B, linefmt="C1-", markerfmt="C1s", basefmt=" ", label='B: "about 7"')
plt.xlabel("x")
plt.ylabel("Membership")
plt.title("Input fuzzy numbers")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 2. Direct implementation of the extension principle

We will support four ordinary binary functions:

- addition: $f(x,y)=x+y$
- maximum: $f(x,y)=\max(x,y)$
- minimum: $f(x,y)=\min(x,y)$
- multiplication: $f(x,y)=xy$

Notice that these functions operate on the **domain values** $x$ and $y$.

For example, $f(x,y)=\max(x,y)$ asks which numerical input is larger. It does **not** mean “take the fuzzy union of $A$ and $B$.”


In [ ]:
def binary_function(x, y, operation):
    if operation == "sum":
        return x + y
    if operation == "max":
        return max(x, y)
    if operation == "min":
        return min(x, y)
    if operation == "multiply":
        return x * y
    raise ValueError(f"Unknown operation: {operation}")


def extension_principle(X, A, B, operation):
    """Apply a binary function to discrete fuzzy sets A and B.

    The output is represented on the same universe X.
    Pairs whose result falls outside X are not represented.
    """
    C = np.zeros(X.size)

    # Map universe values to their array positions.
    position = {value: i for i, value in enumerate(X)}

    for j, x in enumerate(X):
        for k, y in enumerate(X):
            z = binary_function(x, y, operation)

            if z in position:
                candidate = min(A[j], B[k])
                i = position[z]
                C[i] = max(C[i], candidate)

    return C


### Why `min` and then `max`?

Suppose several pairs $(x,y)$ produce the same output $z$.

For each pair, we calculate

$\min(\mu_A(x),\mu_B(y))$.

This gives the degree to which that particular pair supports the output.

Because **any** valid pair can produce $z$, we then take the maximum over all of those possibilities.

That is the computational meaning of

$\mu_C(z)=\max_{f(x,y)=z}\min(\mu_A(x),\mu_B(y))$.


In [ ]:
operation = "sum"
C = extension_principle(X, A, B, operation)

plt.figure(figsize=(11, 5))
plt.stem(X, A, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="A")
plt.stem(X, B, linefmt="C1-", markerfmt="C1s", basefmt=" ", label="B")
plt.stem(X, C, linefmt="C2-", markerfmt="C2^", basefmt=" ", label=f"Result: {operation}")

plt.fill_between(X, C, alpha=0.15)
plt.xlabel("Domain value")
plt.ylabel("Membership")
plt.title(f"Extension principle using operation = {operation!r}")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


### Checkpoint

For addition, the most plausible input pair is $x=3$ and $y=7$.

Therefore,

$z=3+7=10$

and

$\min(\mu_A(3),\mu_B(7))=\min(1,1)=1$.

So the output should have membership 1 at $z=10$.

Check the plot. Does it?


## 3. Try the other functions

Change `operation` below to `"max"`, `"min"`, or `"multiply"`.

Before running each case, predict approximately where the output will be centered.


In [ ]:
operation = "max"   # Try: "sum", "max", "min", "multiply"
C = extension_principle(X, A, B, operation)

nonzero = C > 0

plt.figure(figsize=(10, 4))
plt.stem(X[nonzero], C[nonzero], basefmt=" ")
plt.xlabel("z")
plt.ylabel("Membership")
plt.title(f"Output fuzzy set for f(x,y) = {operation}")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

print("Nonzero output values:")
for z, membership_value in zip(X[nonzero], C[nonzero]):
    print(f"z = {z:2d}   membership = {membership_value:.2f}")


### An important conceptual question: what does `max(A,B)` mean here?

There are two different ideas that are easy to confuse.

**Fuzzy union** combines memberships at the same domain point:

$\mu_{A\cup B}(x)=\max(\mu_A(x),\mu_B(x))$.

But the extension-principle example with `operation = "max"` applies the numerical function

$z=\max(x,y)$

to pairs of **domain values**.

These are different operations and generally produce different fuzzy sets.


In [ ]:
# Compare fuzzy union with the extension of the numerical max function.
fuzzy_union = np.maximum(A, B)
extended_max = extension_principle(X, A, B, "max")

plt.figure(figsize=(10, 4))
plt.stem(X, fuzzy_union, linefmt="C0-", markerfmt="C0o",
         basefmt=" ", label="Fuzzy union: max(A(x), B(x))")
plt.stem(X, extended_max, linefmt="C1-", markerfmt="C1s",
         basefmt=" ", label="Extension of z = max(x,y)")

plt.xlabel("Domain value")
plt.ylabel("Membership")
plt.title("Two different meanings of 'max'")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 4. A limitation of the discrete implementation

Our output array is defined only on the original universe $X=\{1,\ldots,29\}$.

That works for the examples above, but some operations can produce values outside this range. For example, multiplication can produce values much larger than 29.

A more general implementation would construct an output universe appropriate for the selected function.

This is one reason the **$\alpha$-cut approach** is especially useful for fuzzy arithmetic.


# 5. Fuzzy arithmetic with $\alpha$-cuts

An $\alpha$-cut converts a fuzzy number into an ordinary interval.

For a triangular fuzzy number $(a,b,c)$, the $\alpha$-cut is

$[L_\alpha,R_\alpha]$,

where

$L_\alpha=a+(b-a)\alpha$

and

$R_\alpha=c-(c-b)\alpha$.

We will use:

- $A=(1,3,5)$, representing a triangular fuzzy number centered at 3;
- $B=(5,7,9)$, representing a triangular fuzzy number centered at 7.


In [ ]:
A_tri = np.array([1.0, 3.0, 5.0])
B_tri = np.array([5.0, 7.0, 9.0])


def triangular_alpha_cut(fuzzy_number, alpha):
    left, peak, right = fuzzy_number

    lower = left + (peak - left) * alpha
    upper = right - (right - peak) * alpha

    return lower, upper


for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(
        f"alpha = {alpha:.2f}   "
        f"A_alpha = {triangular_alpha_cut(A_tri, alpha)}   "
        f"B_alpha = {triangular_alpha_cut(B_tri, alpha)}"
    )


### Checkpoint: what happens as $\alpha$ increases?

At $\alpha=0$, the interval covers the full support of the triangular fuzzy number.

As $\alpha$ approaches 1, the interval shrinks toward the most plausible value.

For $A=(1,3,5)$, the $\alpha=1$ cut is simply $[3,3]$.


## 6. Interval arithmetic

Once an $\alpha$-cut has converted each fuzzy number into an interval, we can use ordinary interval arithmetic.

If $A_\alpha=[a,b]$ and $B_\alpha=[c,d]$, then:

**Addition**

$A_\alpha+B_\alpha=[a+c,b+d]$.

**Subtraction**

$A_\alpha-B_\alpha=[a-d,b-c]$.

For multiplication, all four endpoint products must be considered:

$A_\alpha B_\alpha=[\min(ac,ad,bc,bd),\max(ac,ad,bc,bd)]$.

Division is similar, but the divisor interval must **not contain zero**.


In [ ]:
def interval_operation(interval_a, interval_b, operation):
    a, b = interval_a
    c, d = interval_b

    if operation == "sum":
        return a + c, b + d

    if operation == "subtract":
        return a - d, b - c

    if operation == "multiply":
        products = [a*c, a*d, b*c, b*d]
        return min(products), max(products)

    if operation == "divide":
        if c <= 0 <= d:
            raise ZeroDivisionError(
                "Division is undefined when the divisor interval contains zero."
            )
        quotients = [a/c, a/d, b/c, b/d]
        return min(quotients), max(quotients)

    raise ValueError(f"Unknown operation: {operation}")


## 7. Visualizing the $\alpha$-cuts

Each horizontal line below is an interval at one $\alpha$ level.

The input cuts are shown together with the result cuts. Reading the result intervals from bottom to top reconstructs the output fuzzy number.

### Prediction

For fuzzy addition of numbers centered at 3 and 7, where should the result reach membership 1?


In [ ]:
num_slices = 30
alpha_cuts = np.linspace(0.0, 1.0, num_slices + 1)

operation = "sum"   # Try: "sum", "multiply", "subtract", "divide"

plt.figure(figsize=(11, 6))

for i, alpha in enumerate(alpha_cuts):
    interval_a = triangular_alpha_cut(A_tri, alpha)
    interval_b = triangular_alpha_cut(B_tri, alpha)
    interval_result = interval_operation(interval_a, interval_b, operation)

    label_a = "A alpha-cuts" if i == 0 else None
    label_b = "B alpha-cuts" if i == 0 else None
    label_r = "Result alpha-cuts" if i == 0 else None

    plt.plot(interval_a, [alpha, alpha], "--", alpha=0.5, label=label_a)
    plt.plot(interval_b, [alpha, alpha], "--", alpha=0.5, label=label_b)
    plt.plot(interval_result, [alpha, alpha], linewidth=2, label=label_r)

plt.xlabel("Domain value")
plt.ylabel("alpha")
plt.title(f"Alpha-cut fuzzy arithmetic: {operation}")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 8. Inspect one $\alpha$-cut by hand

Take $\alpha=0.5$.

For $A=(1,3,5)$,

$A_{0.5}=[2,4]$.

For $B=(5,7,9)$,

$B_{0.5}=[6,8]$.

Therefore, for addition,

$A_{0.5}+B_{0.5}=[2+6,4+8]=[8,12]$.

Use the next cell to verify this result.


In [ ]:
alpha = 0.5

A_cut = triangular_alpha_cut(A_tri, alpha)
B_cut = triangular_alpha_cut(B_tri, alpha)
result_cut = interval_operation(A_cut, B_cut, "sum")

print("A alpha-cut:", A_cut)
print("B alpha-cut:", B_cut)
print("Result:", result_cut)


# Practice

### Exercise 1 — direct extension principle

Using the discrete fuzzy sets $A$ and $B$, predict the output of $z=\min(x,y)$.

Where should the membership reach 1? Run the code to check your prediction.

### Exercise 2 — `max` versus union

Explain in one or two sentences why the extension of $z=\max(x,y)$ is not the same as the fuzzy union $A\cup B$.

### Exercise 3 — $\alpha$-cut arithmetic

For $\alpha=0.5$, use

$A_{0.5}=[2,4]$

and

$B_{0.5}=[6,8]$

to calculate the intervals for:

- $A+B$
- $A-B$
- $A\times B$
- $A/B$

Then verify your answers with `interval_operation()`.

### Exercise 4 — change the fuzzy numbers

Replace $A=(1,3,5)$ with another triangular fuzzy number. Predict how the addition result will move or change width before plotting it.

### Challenge

Why does the direct discrete implementation need an explicit output universe, while the $\alpha$-cut interval calculation does not use one in the same way?


In [ ]:
# Exercise workspace

alpha = 0.5

# Add your calculations and experiments below.


## Summary

The extension principle allows ordinary mathematical functions to act on fuzzy quantities.

For a binary function $z=f(x,y)$, the direct discrete implementation used

$\mu_C(z)=\max_{f(x,y)=z}\min(\mu_A(x),\mu_B(y))$.

The key idea is that many input pairs may support the same output value.

We also used $\alpha$-cuts to convert triangular fuzzy numbers into intervals and then applied interval arithmetic. This often gives a cleaner way to perform fuzzy arithmetic, especially when the output range is not naturally restricted to the original discrete universe.

Finally, remember the distinction between operating on **domain values** and operating directly on **membership values**. That distinction is especially important when interpreting functions such as `max`.
